In [ ]:
# ===============================
# Student Performance Analysis
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# -------------------------------
# Task 1 — Data Exploration
# -------------------------------

print("\n===== TASK 1: DATA EXPLORATION =====\n")

# Load dataset
df = pd.read_csv("students.csv")

# 1. First 5 rows
print("First 5 rows:\n", df.head(), "\n")

# 2. Shape & dtypes
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes, "\n")

# 3. Summary statistics
print("Summary Statistics:\n", df.describe(), "\n")

# 4. Pass/Fail count
print("Pass/Fail Count:\n", df['passed'].value_counts(), "\n")

# 5. Average subject scores (Pass vs Fail)
subject_cols = ['math', 'science', 'english', 'history', 'pe']

pass_avg = df[df['passed'] == 1][subject_cols].mean()
fail_avg = df[df['passed'] == 0][subject_cols].mean()

print("Average Scores (PASS):\n", pass_avg, "\n")
print("Average Scores (FAIL):\n", fail_avg, "\n")

# 6. Highest overall average student
df['overall_avg'] = df[subject_cols].mean(axis=1)
top_student = df.loc[df['overall_avg'].idxmax()]

print("Top Student:", top_student['name'])
print("Average Score:", top_student['overall_avg'], "\n")


# -------------------------------
# Task 2 — Matplotlib Visuals
# -------------------------------

print("\n===== TASK 2: MATPLOTLIB =====\n")

df['avg_score'] = df[subject_cols].mean(axis=1)

# 1. Bar Chart
plt.figure()
df[subject_cols].mean().plot(kind='bar')
plt.title("Average Score per Subject")
plt.xlabel("Subjects")
plt.ylabel("Average Score")
plt.savefig("plot1_bar.png")
plt.show()

# 2. Histogram
plt.figure()
plt.hist(df['math'], bins=5)
mean_math = df['math'].mean()
plt.axvline(mean_math, linestyle='dashed')
plt.title("Math Score Distribution")
plt.xlabel("Math Score")
plt.ylabel("Frequency")
plt.savefig("plot2_hist.png")
plt.show()

# 3. Scatter Plot
plt.figure()
pass_df = df[df['passed'] == 1]
fail_df = df[df['passed'] == 0]

plt.scatter(pass_df['study_hours_per_day'], pass_df['avg_score'], label='Pass')
plt.scatter(fail_df['study_hours_per_day'], fail_df['avg_score'], label='Fail')

plt.title("Study Hours vs Average Score")
plt.xlabel("Study Hours per Day")
plt.ylabel("Average Score")
plt.legend()
plt.savefig("plot3_scatter.png")
plt.show()

# 4. Box Plot
plt.figure()
pass_att = pass_df['attendance_pct'].tolist()
fail_att = fail_df['attendance_pct'].tolist()

plt.boxplot([pass_att, fail_att], labels=['Pass', 'Fail'])
plt.title("Attendance Distribution")
plt.ylabel("Attendance %")
plt.savefig("plot4_box.png")
plt.show()

# 5. Line Plot
plt.figure()
plt.plot(df['name'], df['math'], marker='o', label='Math')
plt.plot(df['name'], df['science'], marker='s', label='Science')

plt.xticks(rotation=45)
plt.title("Math & Science Scores by Student")
plt.xlabel("Student")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig("plot5_line.png")
plt.show()


# -------------------------------
# Task 3 — Seaborn Visuals
# -------------------------------

print("\n===== TASK 3: SEABORN =====\n")

# 1. Bar plots (subplot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

sns.barplot(data=df, x='passed', y='math', ax=ax1)
ax1.set_title("Math vs Pass")

sns.barplot(data=df, x='passed', y='science', ax=ax2)
ax2.set_title("Science vs Pass")

plt.tight_layout()
plt.savefig("plot6_seaborn_bar.png")
plt.show()

# 2. Scatter + Regression
plt.figure()

sns.scatterplot(data=df, x='attendance_pct', y='avg_score', hue='passed')

sns.regplot(data=df[df['passed'] == 1],
            x='attendance_pct', y='avg_score',
            scatter=False, label='Pass')

sns.regplot(data=df[df['passed'] == 0],
            x='attendance_pct', y='avg_score',
            scatter=False, label='Fail')

plt.title("Attendance vs Avg Score")
plt.legend()
plt.savefig("plot7_seaborn_scatter.png")
plt.show()

# 4. Comment
"""
Seaborn is easier for statistical plots like barplots and regression lines,
as it automatically handles grouping and styling. Matplotlib gives more control
but requires more manual work for customization.
"""


# -------------------------------
# Task 4 — Machine Learning
# -------------------------------

print("\n===== TASK 4: MACHINE LEARNING =====\n")

# Features & target
X = df[['math', 'science', 'english', 'history', 'pe',
        'attendance_pct', 'study_hours_per_day']]
y = df['passed']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Training accuracy
train_acc = model.score(X_train_scaled, y_train)
print("Training Accuracy:", train_acc)

# Predictions
y_pred = model.predict(X_test_scaled)

# Test accuracy
test_acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_acc, "\n")

# Detailed results
print("Test Predictions:")
for i, idx in enumerate(X_test.index):
    name = df.loc[idx, 'name']
    actual = y_test.loc[idx]
    pred = y_pred[i]
    result = "✅" if actual == pred else "❌"

    print(f"{name}: Actual={actual}, Predicted={pred} {result}")


# -------------------------------
# Feature Importance
# -------------------------------

print("\nFeature Importance:\n")

coeffs = model.coef_[0]
features = X.columns

feature_importance = sorted(
    zip(features, coeffs),
    key=lambda x: abs(x[1]),
    reverse=True
)

for f, c in feature_importance:
    print(f"{f}: {c:.4f}")

# Plot feature importance
plt.figure()

colors = ['green' if c > 0 else 'red' for _, c in feature_importance]

plt.barh([f for f, _ in feature_importance],
         [c for _, c in feature_importance],
         color=colors)

plt.title("Feature Importance (Logistic Regression)")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.savefig("plot8_feature_importance.png")
plt.show()


# -------------------------------
# Bonus — New Student Prediction
# -------------------------------

print("\n===== BONUS: NEW STUDENT =====\n")

new_student = [[75, 70, 68, 65, 80, 82, 3.2]]

new_scaled = scaler.transform(new_student)

prediction = model.predict(new_scaled)[0]
prob = model.predict_proba(new_scaled)[0]

result = "PASS" if prediction == 1 else "FAIL"

print("Prediction:", result)
print("Probability [Fail, Pass]:", prob)